# Serving: Sichten oder Kopie?

Die Auswertungsumgebung soll **eigene Abfragen** erlauben — jemand sitzt davor,
tippt SQL und wartet auf die Antwort. Dafür gibt es zwei Wege:

| | |
| --- | --- |
| **Sichten** | Eine DuckDB-Datei mit `CREATE VIEW … SELECT * FROM read_parquet(…)`. Sie hält keine Daten, nur Namen. |
| **Kopie** | Dieselben Daten noch einmal als echte Tabellen in der DuckDB-Datei. |

Die Frage ist keine Geschmacksfrage, sondern messbar — genau wie beim
Speicherformat in [Notebook 31](31_storage_evaluation.ipynb).

**Warum sie überhaupt gestellt wurde.** Der Detroit-Branch brachte
`load_duckdb.py` mit, das Bronze in eine DuckDB-Datei importiert. Absprache 7.1
hält aber fest: dateibasierte Schichten als Speicher, **DuckDB als
Abfrageschicht darüber**. Bevor man eine Absprache ändert, misst man besser
nach, was sie kostet.

## 1. Aufbau

Verglichen werden **dieselben Daten** in beiden Formen: die drei
Silver-Tabellen, 410'000 Zeilen zusammen. Beide Varianten entstehen zur
Laufzeit in einem temporären Verzeichnis.

In [ ]:
from pathlib import Path
import glob
import os
import shutil
import statistics
import tempfile
import time

import duckdb

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'data' / 'silver').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'data' / 'silver').exists():
    raise RuntimeError(
        'data/silver nicht gefunden. Zuerst die Silver-Schicht bauen, etwa '
        'python -m silver_pipeline.build_crime_offense'
    )

QUELLEN = {
    p.split('dataset=')[1].split('/')[0]: os.path.abspath(p)
    for p in glob.glob(str(PROJECT_ROOT / 'data' / 'silver' / '**' / '*.parquet'),
                       recursive=True)
}
ARBEIT = Path(tempfile.mkdtemp(prefix='serving-eval-'))

print(f'{"Tabelle":<18}{"Zeilen":>10}{"Parquet":>12}')
gesamt_zeilen = gesamt_bytes = 0
con = duckdb.connect()
for name, pfad in sorted(QUELLEN.items()):
    n = con.execute(f"SELECT count(*) FROM read_parquet('{pfad}')").fetchone()[0]
    b = os.path.getsize(pfad)
    gesamt_zeilen += n; gesamt_bytes += b
    print(f'{name:<18}{n:>10,}{b/1e6:>10.1f} MB')
con.close()
print(f'{"zusammen":<18}{gesamt_zeilen:>10,}{gesamt_bytes/1e6:>10.1f} MB')

## 2. Beide Varianten bauen

In [ ]:
sichten_db = ARBEIT / 'sichten.duckdb'
kopie_db = ARBEIT / 'kopie.duckdb'

con = duckdb.connect(str(sichten_db))
for name, pfad in QUELLEN.items():
    con.execute(f"CREATE VIEW {name} AS SELECT * FROM read_parquet('{pfad}')")
con.close()

con = duckdb.connect(str(kopie_db))
for name, pfad in QUELLEN.items():
    con.execute(f"CREATE TABLE {name} AS SELECT * FROM read_parquet('{pfad}')")
con.close()

print(f'{"Variante":<22}{"Groesse":>12}')
for name, pfad in (('Sichten', sichten_db), ('Kopie', kopie_db)):
    print(f'{name:<22}{os.path.getsize(pfad)/1e6:>10.1f} MB')
print(f'{"Parquet (Quelle)":<22}{gesamt_bytes/1e6:>10.1f} MB')

**Die Sichten-Datei ist winzig**, weil sie nur Namen hält. Die Kopie ist
sogar grösser als die Parquet-Dateien, auf die sie zeigt — die Tract-Geometrie
komprimiert in DuckDBs Format schlechter als in Parquet.

## 3. Die Abfragen

Fünf Muster, wie sie bei einer eigenen Auswertung tatsächlich vorkommen: ein
Filter, zwei Zwei-Wege-Joins, ein Drei-Wege-Join und ein breiter Scan. Gemessen
wird der **erste** Lauf (kalt) und der Median von fünf weiteren.

In [ ]:
ABFRAGEN = {
    'Filter + Gruppierung': '''
        SELECT nibrs_code, count(*) FROM crime_offense
        WHERE city = 'chicago' AND nibrs_group = 'A'
        GROUP BY 1 ORDER BY 2 DESC LIMIT 10''',
    'Join Delikt x Tract (Rate)': '''
        SELECT c.tract_geoid, count(*) f, any_value(t.population) ew,
               count(*) * 100000.0 / nullif(any_value(t.population), 0) rate
        FROM crime_offense c JOIN tract t ON c.tract_geoid = t.geoid
        WHERE c.located GROUP BY 1 HAVING any_value(t.population) > 500
        ORDER BY rate DESC LIMIT 20''',
    'Join Delikt x Wetter': '''
        SELECT w.city, count(*) FROM crime_offense c
        JOIN weather_day w ON c.city = w.city AND c.occurred_date_local = w.date
        WHERE w.temp_max_c > 30 GROUP BY 1''',
    'Drei-Wege-Join': '''
        SELECT t.geoid, w.city, count(*) FROM crime_offense c
        JOIN tract t ON c.tract_geoid = t.geoid
        JOIN weather_day w ON c.city = w.city AND c.occurred_date_local = w.date
        WHERE w.temp_max_c > 25 GROUP BY 1, 2 ORDER BY 3 DESC LIMIT 20''',
    'Breiter Scan': '''
        SELECT city, nibrs_description, source_offense_description, tract_geoid,
               occurred_at_utc FROM crime_offense WHERE arrest
        ORDER BY occurred_at_utc LIMIT 500''',
}

def messen(datenbank, sql, laeufe=5):
    con = duckdb.connect(str(datenbank), read_only=True)
    t = time.perf_counter(); con.execute(sql).fetchall(); kalt = time.perf_counter() - t
    zeiten = []
    for _ in range(laeufe):
        t = time.perf_counter(); con.execute(sql).fetchall()
        zeiten.append(time.perf_counter() - t)
    con.close()
    return kalt, statistics.median(zeiten)

print(f'{"Abfrage":<30}{"Sichten":>20}{"Kopie":>20}{"Faktor":>9}')
print(f'{"":<30}{"kalt / warm":>20}{"kalt / warm":>20}')
for name, sql in ABFRAGEN.items():
    s_kalt, s_warm = messen(sichten_db, sql)
    k_kalt, k_warm = messen(kopie_db, sql)
    print(f'{name:<30}{s_kalt*1000:>9.0f} /{s_warm*1000:>7.1f}ms'
          f'{k_kalt*1000:>9.0f} /{k_warm*1000:>7.1f}ms{s_warm/k_warm:>8.1f}x')

**Die Sichten sind nicht langsamer — teils schneller.** Alle Abfragen
liegen im einstelligen Millisekundenbereich, auch kalt. DuckDB liest Parquet
nativ mit Projektions- und Prädikat-Pushdown; bei 36 MB über drei Tabellen gibt
es keinen Dekodier-Aufwand, der sich gegen den Vorteil der Kopie durchsetzen
würde.

Ein Mensch kann den Unterschied zwischen 3,2 ms und 3,6 ms nicht wahrnehmen.
Die Entscheidung fällt also **nicht** über Geschwindigkeit.

## 4. Liefern beide dasselbe?

Ohne diese Prüfung wäre die Messung wertlos — eine schnelle Variante, die etwas
anderes ausrechnet, ist keine Alternative.

In [ ]:
PRUEFUNGEN = {
    'Zeilen crime_offense': 'SELECT count(*) FROM crime_offense',
    'Join x Tract': '''SELECT count(*), sum(f) FROM (
        SELECT c.tract_geoid, count(*) f FROM crime_offense c
        JOIN tract t ON c.tract_geoid = t.geoid GROUP BY 1)''',
    'Join x Wetter': '''SELECT count(*) FROM crime_offense c
        JOIN weather_day w ON c.city = w.city AND c.occurred_date_local = w.date
        WHERE w.temp_max_c > 30''',
    'Verteilung je NIBRS-Code': '''SELECT nibrs_code, count(*)
        FROM crime_offense GROUP BY 1 ORDER BY 1''',
}

for name, sql in PRUEFUNGEN.items():
    a = duckdb.connect(str(sichten_db), read_only=True).execute(sql).fetchall()
    b = duckdb.connect(str(kopie_db), read_only=True).execute(sql).fetchall()
    print(f'  {name:<26}{"identisch" if a == b else "ABWEICHUNG"}')

## 5. Was den Ausschlag gibt

Nicht die Geschwindigkeit, sondern drei Eigenschaften, die sich nicht messen
lassen, sondern aus der Sache folgen:

**Eine Kopie kann veralten.** Nach einem Silver-Neubau sind Sichten sofort
aktuell. Eine kopierte Tabelle zeigt den alten Stand, bis jemand neu importiert
— und man sieht ihr nicht an, dass sie es tut. Die nächste Zelle zeigt das.

In [ ]:
probe = ARBEIT / 'probe'
probe.mkdir(exist_ok=True)
quelle = probe / 'daten.parquet'

con = duckdb.connect()
con.execute(f"COPY (SELECT 1 AS wert) TO '{quelle}' (FORMAT PARQUET)")
con.close()

con = duckdb.connect(str(probe / 'beide.duckdb'))
con.execute(f"CREATE VIEW als_sicht AS SELECT * FROM read_parquet('{quelle}')")
con.execute(f"CREATE TABLE als_kopie AS SELECT * FROM read_parquet('{quelle}')")
print('vorher   Sicht:', con.execute('SELECT count(*) FROM als_sicht').fetchone()[0],
      ' Kopie:', con.execute('SELECT count(*) FROM als_kopie').fetchone()[0])

# Die Quelle wird neu gebaut - so wie Silver nach einem neuen Lauf
con2 = duckdb.connect()
con2.execute(f"COPY (SELECT * FROM range(1, 101) t(wert)) TO '{quelle}' (FORMAT PARQUET)")
con2.close()

print('nachher  Sicht:', con.execute('SELECT count(*) FROM als_sicht').fetchone()[0],
      ' Kopie:', con.execute('SELECT count(*) FROM als_kopie').fetchone()[0],
      '  <- die Kopie zeigt weiter den alten Stand')
con.close()

**Eine Kopie ist eine zweite Wahrheit.** Absprache 7.1 hält die
dateibasierten Schichten als Speicher fest, DuckDB als Abfrageschicht darüber.
Eine Kopie von Bronze oder Silver in einer veränderbaren Datei bricht das:
Dann gibt es zwei Orte, an denen dieselbe Zahl steht, und keine Regel, welcher
gilt.

**Eine Kopie kostet Platz ohne Gegenwert** — hier das rund 150-fache, bei
gleicher Geschwindigkeit.

## 6. Und die Bequemlichkeit?

Das eigentliche Problem bei eigenen Abfragen ist nicht Tempo, sondern dass
niemand das hier tippen will:

```sql
SELECT * FROM read_parquet('data/silver/dataset=crime_offense/part-0001.parquet')
```

Genau das lösen die Sichten. Sie geben den Tabellen Namen, ohne die Daten zu
kopieren — und **eigene Tabellen gehen trotzdem**, weil die Datei eine echte
Datenbank ist.

In [ ]:
con = duckdb.connect(str(sichten_db))
con.execute('''CREATE OR REPLACE TABLE meine_auswertung AS
    SELECT city, nibrs_description, count(*) AS faelle
    FROM crime_offense WHERE nibrs_group = 'A'
    GROUP BY 1, 2 ORDER BY 3 DESC''')
print('eigene Tabelle:',
      con.execute('SELECT count(*) FROM meine_auswertung').fetchone()[0], 'Zeilen')

arten = dict(con.execute('''SELECT table_type, count(*) FROM information_schema.tables
    WHERE table_schema = 'main' GROUP BY 1''').fetchall())
print('Inhalt der Datei:', arten, ' <- Sichten und die eigene Tabelle nebeneinander')
con.close()

## 7. Ergebnis

**Entschieden: Sichten.** Nicht weil sie schneller sind — das sind sie nur
knapp —, sondern weil die Kopie auf jeder anderen Achse verliert: Sie kann
veralten, sie ist eine zweite Wahrheit neben den Schichten, und sie kostet das
150-fache an Platz.

Gebaut wird die Datei von
[`src/crime/serving/build_analysis_db.py`](../src/crime/serving/build_analysis_db.py):

```bash
python -m serving.build_analysis_db
duckdb data/serving/analyse.duckdb
```

Sie ist abgeleitet und nicht versioniert — wie Silver selbst.

**Grenzen dieser Messung.** 410'000 Zeilen, 36 MB, warmer Betriebssystem-Cache,
ein Rechner, ein Leser. Bei hundertfacher Datenmenge könnte die Antwort kippen;
für dieses Projekt nicht. Und DuckDB kennt nur einen Schreiber: Legen zwei
Personen gleichzeitig eigene Tabellen in derselben Datei an, kollidiert das —
dann besser je eine eigene Datei mit `ATTACH`.

In [ ]:
shutil.rmtree(ARBEIT, ignore_errors=True)
print('Arbeitsverzeichnis entfernt:', ARBEIT)